# Method-wise sampling results

This notebook writes five main figures plus one diagnostic figure: one standalone Figure A for the proxy landscape, four Figure B method-wise sampling-result panels, and one computed-barrier/corridor diagnostic plot.

- Figure A: proxy landscape, shaded geometry, and registered regions only.
- Figure B1/B2/B3/B4: method-wise sampling results, one method per figure.
- Diagnostic shaded-guide plot: shows the minimax grid paths used to compute the yellow corridor and gray barrier windows.
- Figure A and Figure B1-B4 call the same `computed_shaded_segments()` output as the diagnostic plot.
- HMC and pL are displayed as sampling trajectories.
- Random-walk MC and vMF+L2 are displayed as point samples.
- All panels use the same proxy landscape, registered regions, inverse temperature beta, domain, and QC definitions. The visible advantage or disadvantage should therefore come from the sampler or proposal mechanism, not from changing the target distribution.

Shaded overlays used below:

- The yellow shaded band is the computed minimax path between the solution core and the nearby same-valley basin.
- The gray shaded bands are high-energy windows around the peak of computed minimax paths to the across-barrier and remote-needle regions. They are not confidence intervals or sampler output; they are extracted from the proxy energy grid.

In [ ]:
from pathlib import Path
import heapq
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Patch


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "weakens_benchmark").exists():
            return candidate
    raise RuntimeError("Could not find project root from current working directory")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from weakens_benchmark.config import load_config
from weakens_benchmark.importance import normalized_weights
from weakens_benchmark.landscape import ProxyLandscape

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 240,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
CONFIG_PATH = ROOT / "01_dataset_proxy" / "config" / "default.json"
cfg = load_config(CONFIG_PATH)
experiment_id = str(cfg["experiment_id"])
landscape = ProxyLandscape(cfg)

dataset_dir = ROOT / "01_dataset_proxy" / "raw_outputs" / experiment_id
baseline_dir = ROOT / "03_baseline_samplers" / "raw_outputs" / experiment_id
vmf_dir = ROOT / "04_vmf_l2_importance" / "raw_outputs" / experiment_id
qc_dir = ROOT / "05_qc_and_figures" / "raw_outputs" / experiment_id
out_dir = ROOT / "05_qc_and_figures" / "figures" / experiment_id / "method_sampling_results"
out_dir.mkdir(parents=True, exist_ok=True)
for stale_png in out_dir.glob(f"{experiment_id}_*.png"):
    stale_png.unlink()

grid_npz = np.load(dataset_dir / "grid_reference.npz")
x = grid_npz["x"]
y = grid_npz["y"]
xx, yy = np.meshgrid(x, y)
energy = grid_npz["energy"]

baseline_npz = np.load(baseline_dir / "baseline_samples.npz")
vmf_npz = np.load(vmf_dir / "vmf_l2_samples.npz")
qc_df = pd.read_csv(qc_dir / "qc_summary.csv")

with open(baseline_dir / "baseline_reproduction_metadata.json", encoding="utf-8") as fh:
    baseline_meta = json.load(fh)
with open(vmf_dir / "importance_summary.json", encoding="utf-8") as fh:
    vmf_meta = json.load(fh)

print(f"experiment_id: {experiment_id}")
print(f"shared beta / inverse temperature: {cfg['beta']}")
print(f"shared initial point for local baselines: {cfg['samplers']['initial']}")
print(f"output directory: {out_dir}")

## Constant target and method-specific mechanisms

The target density is always proportional to `exp(-beta * E(z))` with `beta = cfg["beta"]`. HMC, pL, MC, and vMF+L2 all read the same landscape object and the same region definitions. What changes is the sampling mechanism:

- MC: local Gaussian random-walk Metropolis retained points.
- HMC: full-gradient leapfrog trajectory on the same target energy.
- pL: underdamped minibatch Langevin trajectory on the same target scale.
- vMF+L2: directional/radial proposal plus importance reweighting using the same `-beta * E(z)` target log density.

In [ ]:
METHOD_COLORS = {
    "random_walk_mcmc": "#C44E52",
    "hmc": "#4C72B0",
    "pseudo_langevin": "#DD8452",
    "vmf_l2_final": "#2E8B57",
}

REGION_COLORS = {
    "solution_core": "#4C72B0",
    "near_same_valley": "#CCB974",
    "across_barrier": "#8172B3",
    "remote_needle": "#64B5CD",
}

METHOD_DISPLAY = {
    "random_walk_mcmc": "MC retained points",
    "hmc": "HMC sampling trajectory",
    "pseudo_langevin": "pL sampling trajectory",
    "vmf_l2_final": "vMF+L2 weighted points",
}

METHOD_FIGURE_ID = {
    "random_walk_mcmc": "Figure B1",
    "hmc": "Figure B2",
    "pseudo_langevin": "Figure B3",
    "vmf_l2_final": "Figure B4",
}

METHOD_SLUG = {
    "random_walk_mcmc": "figure_B1_mc_points",
    "hmc": "figure_B2_hmc_sampling_traj",
    "pseudo_langevin": "figure_B3_pl_sampling_traj",
    "vmf_l2_final": "figure_B4_vmf_l2_weighted_points",
}


def coarse_path(samples: np.ndarray, max_points: int = 180) -> np.ndarray:
    samples = np.asarray(samples, dtype=np.float64)
    if samples.ndim != 2 or samples.shape[0] == 0:
        return np.empty((0, 2), dtype=np.float64)
    if samples.shape[0] > max_points:
        idx = np.linspace(0, samples.shape[0] - 1, max_points).astype(int)
        samples = samples[idx]
    if samples.shape[0] >= 7:
        kernel = np.ones(5, dtype=np.float64) / 5.0
        samples = np.column_stack([
            np.convolve(samples[:, 0], kernel, mode="same"),
            np.convolve(samples[:, 1], kernel, mode="same"),
        ])
    return samples


def thin_points(samples: np.ndarray, max_points: int = 4500) -> np.ndarray:
    samples = np.asarray(samples, dtype=np.float64)
    if samples.shape[0] <= max_points:
        return samples
    idx = np.linspace(0, samples.shape[0] - 1, max_points).astype(int)
    return samples[idx]


def weighted_vmf_points(max_points: int = 6500) -> tuple[np.ndarray, np.ndarray]:
    samples = vmf_npz["samples"]
    log_weights = vmf_npz["log_weights"]
    finite_idx = np.flatnonzero(np.isfinite(log_weights))
    weights = normalized_weights(log_weights[finite_idx])
    rng = np.random.default_rng(20260623)
    take_n = min(max_points, finite_idx.size)
    take_local = rng.choice(finite_idx.size, size=take_n, replace=False, p=weights / weights.sum())
    idx = finite_idx[take_local]
    return samples[idx], weights[take_local]


def draw_shaded_geometry(ax: plt.Axes) -> None:
    for segment in computed_shaded_segments():
        pts = segment["points"]
        ax.plot(
            pts[:, 0],
            pts[:, 1],
            color=segment["color"],
            lw=segment["lw"],
            alpha=segment["alpha"],
            solid_capstyle="round",
            zorder=4,
        )


def draw_regions(ax: plt.Axes) -> None:
    for basin in landscape.basins:
        color = REGION_COLORS.get(basin.name, "#333333")
        patch = Ellipse(
            xy=basin.center,
            width=float(2.0 * basin.axes[0] * basin.region_radius * 0.88),
            height=float(2.0 * basin.axes[1] * basin.region_radius * 0.88),
            angle=float(np.degrees(basin.angle)),
            facecolor=color,
            edgecolor=color,
            lw=1.8,
            alpha=0.22,
            zorder=5,
        )
        ax.add_patch(patch)


def draw_landscape(ax: plt.Axes) -> None:
    cap = np.nanquantile(energy[np.isfinite(energy)], 0.96)
    ax.contourf(xx, yy, np.minimum(energy, cap), levels=26, cmap="Greys", alpha=0.22)
    ax.contour(xx, yy, energy, levels=11, colors="black", linewidths=0.18, alpha=0.13)
    draw_shaded_geometry(ax)
    draw_regions(ax)
    ax.set_xlim(landscape.xlim)
    ax.set_ylim(landscape.ylim)
    ax.set_xlabel("collective coordinate 1")
    ax.set_ylabel("collective coordinate 2")


def region_handles() -> list[Patch]:
    handles = []
    for idx, basin in enumerate(landscape.basins, start=1):
        color = REGION_COLORS.get(basin.name, "#333333")
        handles.append(
            Patch(
                facecolor=color,
                edgecolor=color,
                alpha=0.35,
                label=f"R{idx} {basin.name.replace('_', ' ')}",
            )
        )
    return handles


def plot_figure_a(save_path: Path | None = None):
    fig, ax = plt.subplots(figsize=(7.2, 5.3))
    fig.subplots_adjust(top=0.82, bottom=0.31, left=0.12, right=0.92)
    cap = np.nanquantile(energy[np.isfinite(energy)], 0.96)
    im = ax.contourf(xx, yy, np.minimum(energy, cap), levels=44, cmap="magma_r")
    ax.contour(xx, yy, energy, levels=12, colors="black", linewidths=0.25, alpha=0.25)
    draw_shaded_geometry(ax)
    draw_regions(ax)
    ax.set_xlim(landscape.xlim)
    ax.set_ylim(landscape.ylim)
    ax.set_xlabel("collective coordinate 1")
    ax.set_ylabel("collective coordinate 2")
    cbar = fig.colorbar(im, ax=ax, fraction=0.047, pad=0.018)
    cbar.set_label("proxy energy")
    fig.suptitle("Figure A. Proxy landscape and registered regions", fontsize=13, weight="bold")
    fig.text(0.12, 0.86, f"shared beta={cfg['beta']}; same domain and regions for all methods", ha="left", va="center", fontsize=10)
    fig.legend(
        handles=[*region_handles(), *shaded_handles()],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        frameon=False,
        fontsize=8.4,
        ncols=2,
    )
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    return fig


_COMPUTED_SHADED_SEGMENTS = None
_COMPUTED_SHADED_METADATA = None


def basin_center(name: str) -> np.ndarray:
    for basin in landscape.basins:
        if basin.name == name:
            return np.asarray(basin.center, dtype=np.float64)
    raise KeyError(name)


def nearest_grid_ij(point: np.ndarray) -> tuple[int, int]:
    point = np.asarray(point, dtype=np.float64)
    ix = int(np.argmin(np.abs(x - point[0])))
    iy = int(np.argmin(np.abs(y - point[1])))
    return iy, ix


def minimax_path(start_name: str, target_name: str) -> dict:
    start = nearest_grid_ij(basin_center(start_name))
    target = nearest_grid_ij(basin_center(target_name))
    h, w = energy.shape
    best = np.full((h, w), np.inf, dtype=np.float64)
    prev_y = np.full((h, w), -1, dtype=np.int32)
    prev_x = np.full((h, w), -1, dtype=np.int32)
    best[start] = float(energy[start])
    heap = [(float(best[start]), start[0], start[1])]
    neighbors = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1),
        (-1, -1),
        (-1, 1),
        (1, -1),
        (1, 1),
    ]
    while heap:
        cost, iy, ix = heapq.heappop(heap)
        if cost != best[iy, ix]:
            continue
        if (iy, ix) == target:
            break
        for dy, dx in neighbors:
            ny = iy + dy
            nx = ix + dx
            if ny < 0 or ny >= h or nx < 0 or nx >= w:
                continue
            new_cost = max(cost, float(energy[ny, nx]))
            if new_cost < best[ny, nx]:
                best[ny, nx] = new_cost
                prev_y[ny, nx] = iy
                prev_x[ny, nx] = ix
                heapq.heappush(heap, (new_cost, ny, nx))

    cells = []
    iy, ix = target
    while iy >= 0 and ix >= 0:
        cells.append((iy, ix))
        if (iy, ix) == start:
            break
        py = int(prev_y[iy, ix])
        px = int(prev_x[iy, ix])
        iy, ix = py, px
    cells = cells[::-1]
    iys = np.asarray([c[0] for c in cells], dtype=np.int32)
    ixs = np.asarray([c[1] for c in cells], dtype=np.int32)
    path = np.column_stack([x[ixs], y[iys]])
    path_energy = energy[iys, ixs].astype(np.float64)
    return {
        "start": start_name,
        "target": target_name,
        "path": path,
        "energy": path_energy,
        "minimax_energy": float(np.max(path_energy)),
    }


def smooth_polyline(points: np.ndarray, window: int = 9) -> np.ndarray:
    points = np.asarray(points, dtype=np.float64)
    if points.shape[0] < window:
        return points
    if window % 2 == 0:
        window += 1
    kernel = np.ones(window, dtype=np.float64) / float(window)
    out = np.column_stack([
        np.convolve(points[:, 0], kernel, mode="same"),
        np.convolve(points[:, 1], kernel, mode="same"),
    ])
    half = window // 2
    out[:half] = points[:half]
    out[-half:] = points[-half:]
    return out


def thin_polyline(points: np.ndarray, max_points: int = 90) -> np.ndarray:
    if points.shape[0] <= max_points:
        return points
    idx = np.unique(np.linspace(0, points.shape[0] - 1, max_points).astype(int))
    return points[idx]


def barrier_window(path: np.ndarray, path_energy: np.ndarray, fraction: float = 0.62) -> tuple[np.ndarray, dict]:
    peak_idx = int(np.argmax(path_energy))
    endpoint_floor = float(max(path_energy[0], path_energy[-1]))
    peak_energy = float(path_energy[peak_idx])
    threshold = endpoint_floor + fraction * max(peak_energy - endpoint_floor, 0.0)
    if peak_energy <= endpoint_floor + 1.0e-9:
        threshold = float(np.quantile(path_energy, 0.80))
    mask = path_energy >= threshold
    lo = peak_idx
    hi = peak_idx
    while lo > 0 and mask[lo - 1]:
        lo -= 1
    while hi + 1 < path_energy.size and mask[hi + 1]:
        hi += 1
    lo = max(0, lo - 4)
    hi = min(path_energy.size - 1, hi + 4)
    if hi <= lo:
        lo = max(0, peak_idx - 3)
        hi = min(path_energy.size - 1, peak_idx + 3)
    segment = path[lo : hi + 1]
    meta = {
        "peak_idx": peak_idx,
        "peak_x": float(path[peak_idx, 0]),
        "peak_y": float(path[peak_idx, 1]),
        "endpoint_floor": endpoint_floor,
        "peak_energy": peak_energy,
        "threshold_energy": float(threshold),
        "barrier_height_over_endpoint_floor": float(peak_energy - endpoint_floor),
        "window_start_idx": int(lo),
        "window_end_idx": int(hi),
    }
    return segment, meta


def compute_shaded_geometry() -> tuple[list[dict], list[dict]]:
    specs = [
        {
            "guide": "C1",
            "kind": "corridor",
            "start": "solution_core",
            "target": "near_same_valley",
            "short": "computed R1->R2 low-loss minimax corridor",
            "name": "computed yellow corridor: R1 solution core to R2 same-valley basin",
            "color": "#F2D887",
            "lw": 13.0,
            "alpha": 0.26,
        },
        {
            "guide": "B2",
            "kind": "barrier",
            "start": "near_same_valley",
            "target": "across_barrier",
            "short": "computed barrier window on R2->R3 minimax path",
            "name": "computed gray barrier: high-energy window on R2 to R3 minimax path",
            "color": "#606060",
            "lw": 6.0,
            "alpha": 0.34,
        },
        {
            "guide": "B3",
            "kind": "barrier",
            "start": "solution_core",
            "target": "remote_needle",
            "short": "computed barrier window on R1->R4 minimax path",
            "name": "computed gray barrier: high-energy window on R1 to R4 minimax path",
            "color": "#606060",
            "lw": 5.2,
            "alpha": 0.28,
        },
    ]
    segments = []
    metadata = []
    for spec in specs:
        path_record = minimax_path(spec["start"], spec["target"])
        path = path_record["path"]
        path_energy = path_record["energy"]
        if spec["kind"] == "corridor":
            segment_points = path
            peak_idx = int(np.argmax(path_energy))
            meta = {
                "peak_idx": peak_idx,
                "peak_x": float(path[peak_idx, 0]),
                "peak_y": float(path[peak_idx, 1]),
                "endpoint_floor": float(max(path_energy[0], path_energy[-1])),
                "peak_energy": float(np.max(path_energy)),
                "threshold_energy": np.nan,
                "barrier_height_over_endpoint_floor": float(np.max(path_energy) - max(path_energy[0], path_energy[-1])),
                "window_start_idx": 0,
                "window_end_idx": int(path.shape[0] - 1),
            }
        else:
            segment_points, meta = barrier_window(path, path_energy)
        segment_points = smooth_polyline(thin_polyline(segment_points, max_points=120), window=7)
        full_path = smooth_polyline(thin_polyline(path, max_points=180), window=7)
        segment = {
            **spec,
            "points": segment_points,
            "full_path": full_path,
            "path_energy": path_energy,
        }
        segments.append(segment)
        metadata.append({
            "guide": spec["guide"],
            "kind": spec["kind"],
            "start": spec["start"],
            "target": spec["target"],
            "path_nodes": int(path.shape[0]),
            "segment_nodes": int(segment_points.shape[0]),
            "minimax_energy": float(path_record["minimax_energy"]),
            **meta,
        })
    return segments, metadata


def computed_shaded_segments() -> list[dict]:
    global _COMPUTED_SHADED_SEGMENTS, _COMPUTED_SHADED_METADATA
    if _COMPUTED_SHADED_SEGMENTS is None:
        _COMPUTED_SHADED_SEGMENTS, _COMPUTED_SHADED_METADATA = compute_shaded_geometry()
    return _COMPUTED_SHADED_SEGMENTS


def computed_shaded_frame() -> pd.DataFrame:
    global _COMPUTED_SHADED_SEGMENTS, _COMPUTED_SHADED_METADATA
    if _COMPUTED_SHADED_METADATA is None:
        computed_shaded_segments()
    return pd.DataFrame(_COMPUTED_SHADED_METADATA)


def plot_computed_barrier_guides(save_path: Path | None = None):
    fig, ax = plt.subplots(figsize=(7.2, 5.3))
    fig.subplots_adjust(top=0.80, bottom=0.34, left=0.12, right=0.96)
    cap = np.nanquantile(energy[np.isfinite(energy)], 0.96)
    ax.contourf(xx, yy, np.minimum(energy, cap), levels=32, cmap="Greys", alpha=0.18)
    ax.contour(xx, yy, energy, levels=11, colors="black", linewidths=0.18, alpha=0.16)
    draw_regions(ax)

    handles = []
    for segment in computed_shaded_segments():
        pts = segment["points"]
        full_path = segment["full_path"]
        ax.plot(full_path[:, 0], full_path[:, 1], color=segment["color"], lw=1.2, alpha=0.70, linestyle=(0, (3, 3)), zorder=5)
        ax.plot(
            pts[:, 0],
            pts[:, 1],
            color=segment["color"],
            lw=segment["lw"],
            alpha=segment["alpha"],
            solid_capstyle="round",
            zorder=6,
        )
        if segment["kind"] == "barrier":
            path = segment["full_path"]
            meta = computed_shaded_frame().set_index("guide").loc[segment["guide"]]
            ax.scatter([meta["peak_x"]], [meta["peak_y"]], marker="^", s=66, color="#222222", edgecolors="white", linewidths=0.8, zorder=9)
        mid = pts[len(pts) // 2]
        ax.text(mid[0], mid[1] + 0.24, segment["guide"], ha="center", va="bottom", fontsize=10, weight="bold", color="#202020", zorder=9)
        handles.append(Line2D([0], [0], color=segment["color"], lw=4, alpha=0.78, label=f"{segment['guide']}: {segment['short']}"))

    handles.append(Line2D([0], [0], color="#555555", lw=1.2, linestyle=(0, (3, 3)), label="dashed: full minimax path"))
    handles.append(Line2D([0], [0], marker="^", color="#222222", lw=0, markersize=7, label="triangle: peak energy on barrier path"))
    ax.set_xlim(landscape.xlim)
    ax.set_ylim(landscape.ylim)
    ax.set_xlabel("collective coordinate 1")
    ax.set_ylabel("collective coordinate 2")
    fig.suptitle("Diagnostic. Computed minimax corridor and barrier windows", fontsize=13, weight="bold")
    fig.text(0.12, 0.855, "Gray windows are extracted around the energy peak of minimax paths on the proxy grid.", ha="left", va="center", fontsize=10)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.025), frameon=False, fontsize=8.2, ncols=1)
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    return fig


def qc_label(method: str) -> str:
    row = qc_df.set_index("method").loc[method]
    return (
        f"coverage {int(row['covered_important_regions'])}/{int(row['important_regions'])}, "
        f"L1 {float(row['region_l1_error']):.3f}, ESS {float(row['ess_fraction']):.3f}"
    )


def shaded_handles() -> list[Patch]:
    return [
        Patch(
            facecolor="#F2D887",
            edgecolor="#D8BE72",
            alpha=0.42,
            label="computed yellow band: R1->R2 minimax low-loss corridor",
        ),
        Patch(
            facecolor="#606060",
            edgecolor="#606060",
            alpha=0.30,
            label="computed gray bands: high-energy windows on minimax paths",
        ),
    ]


def method_handle(method: str):
    color = METHOD_COLORS[method]
    if method in {"hmc", "pseudo_langevin"}:
        return Line2D([0], [0], color=color, lw=2.2, label=METHOD_DISPLAY[method])
    return Line2D([0], [0], marker="o", color=color, lw=0, markersize=6, label=METHOD_DISPLAY[method])

In [ ]:
def plot_method_result(method: str, ax: plt.Axes | None = None, save_path: Path | None = None):
    created = ax is None
    if created:
        fig, ax = plt.subplots(figsize=(7.2, 5.3))
        fig.subplots_adjust(top=0.82, bottom=0.27, left=0.12, right=0.96)
    else:
        fig = ax.figure

    draw_landscape(ax)
    color = METHOD_COLORS[method]

    if method == "random_walk_mcmc":
        pts = thin_points(baseline_npz["random_walk_mcmc_samples"], max_points=5000)
        ax.scatter(pts[:, 0], pts[:, 1], s=8, color=color, alpha=0.34, edgecolors="none", zorder=8)
    elif method == "vmf_l2_final":
        pts, weights = weighted_vmf_points(max_points=6500)
        denom = max(float(np.quantile(weights, 0.98)), 1.0e-300)
        sizes = 7.0 + 120.0 * np.minimum(weights / denom, 1.0)
        ax.scatter(pts[:, 0], pts[:, 1], s=sizes, color=color, alpha=0.24, edgecolors="none", zorder=8)
    elif method in {"hmc", "pseudo_langevin"}:
        pts = baseline_npz[f"{method}_samples"]
        path = coarse_path(pts, max_points=190)
        ax.plot(path[:, 0], path[:, 1], color=color, lw=2.0, alpha=0.84, zorder=8)
        ax.scatter(path[:1, 0], path[:1, 1], s=42, color=color, edgecolors="white", linewidths=0.8, zorder=9)
        end = path[-1]
        ax.plot([end[0] - 0.08, end[0] + 0.08], [end[1] - 0.08, end[1] + 0.08], color=color, lw=2.2, zorder=9)
        ax.plot([end[0] - 0.08, end[0] + 0.08], [end[1] + 0.08, end[1] - 0.08], color=color, lw=2.2, zorder=9)
    else:
        raise KeyError(method)

    if created:
        fig.suptitle(f"{METHOD_FIGURE_ID[method]}. {METHOD_DISPLAY[method]} on the shared beta={cfg['beta']} landscape", fontsize=13, weight="bold")
        fig.text(0.12, 0.86, qc_label(method), ha="left", va="center", fontsize=10)
        fig.legend(
            handles=[method_handle(method), *shaded_handles()],
            loc="lower center",
            bbox_to_anchor=(0.5, 0.02),
            frameon=False,
            fontsize=8.5,
            ncols=1,
        )
    else:
        ax.set_title(f"{METHOD_DISPLAY[method]}\n{qc_label(method)}", fontsize=10.5)

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    return fig, ax


figure_paths = []

figure_a_path = out_dir / f"{experiment_id}_figure_A_proxy_landscape_regions.png"
fig = plot_figure_a(save_path=figure_a_path)
display(fig)
plt.close(fig)
figure_paths.append(figure_a_path)

for method in ["random_walk_mcmc", "hmc", "pseudo_langevin", "vmf_l2_final"]:
    out_path = out_dir / f"{experiment_id}_{METHOD_SLUG[method]}.png"
    fig, _ = plot_method_result(method, save_path=out_path)
    display(fig)
    plt.close(fig)
    figure_paths.append(out_path)

diagnostic_paths = []
computed_guides_path = out_dir / f"{experiment_id}_diagnostic_computed_barrier_guides.png"
fig = plot_computed_barrier_guides(save_path=computed_guides_path)
display(fig)
plt.close(fig)
diagnostic_paths.append(computed_guides_path)

display(computed_shaded_frame())
figure_paths + diagnostic_paths

In [ ]:
assert len(figure_paths) == 5
assert len(diagnostic_paths) == 1
missing = [path for path in [*figure_paths, *diagnostic_paths] if not path.exists()]
assert not missing, missing
pd.DataFrame({
    "figure": ["A", "B1", "B2", "B3", "B4", "diagnostic"],
    "path": [str(path) for path in [*figure_paths, *diagnostic_paths]],
})